In [1]:
pip install scapy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\priya\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
##install necessary genai extension
!pip install scapy google-genai

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.2 -> 26.2.1


[notice] To update, run: C:\Users\priya\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

In [ ]:
from scapy.all import rdpcap, IP, TCP, UDP, DNSQR
from google import genai
from google.genai import types
import os

def parse_pcap(logs):
    connections = set() #new empty set for source and destination IPs, ports and protocols 
    dns_queries = set() #new empty set for dns queries 
    packets = rdpcap(logs) #dedicated packet parsing library in Scapy to read scrambled, unreadable characters as PCAPs are compiled binary files

    for pkt in packets:

        # collecting source and destination (IP & ports)  based on the protocol like TCP/UDP as port number vary in different protocols
        if IP in pkt:
                src = pkt[IP].src
                dst = pkt[IP].dst
                if TCP in pkt:
                    connections.add(f"TCP: {src}:{pkt[TCP].sport} -> {dst}:{pkt[TCP].dport}")
                elif UDP in pkt:
                    connections.add(f"UDP: {src}:{pkt[UDP].sport} -> {dst}:{pkt[UDP].dport}")

        # collecting DNS Queries 
        if DNSQR in pkt:
            try: # using exceptional handling as every packet may not have a dns query
                # print(pkt[DNSQR].qname)
                # print(pkt[DNSQR].qname.decode('utf-8', errors='ignore'))
                # print(pkt[DNSQR].qname.decode('utf-8', errors='ignore').strip('.'))
                # print("*"*100)
                query = pkt[DNSQR].qname.decode('utf-8', errors='ignore').strip('.') # decode(utf8) translate raw domain back into a normal text string
                dns_queries.add(query)
            except:
                pass

    #Pushing extracted IP, port and DNS query into variable extracted_data
    extracted_data = "### Unique Network Connections ###\n"
    extracted_data += "\n".join(list(connections)) 
    extracted_data += "\n".join(list(dns_queries))

    return extracted_data

def AI_enrichment(extracted_data):
    client = genai.Client(api_key="Enter Gemini API Key")
    
    prompt = """
    You are an expert Cybersecurity SOC Analyst. Review the following network traffic summary extracted from a PCAP file.
    Identify any anomalous, suspicious, or malicious connections that could indicate malware Command and Control (C2) beaconing, data exfiltration, or a user clicking a phishing link.
    
    Focus on:
    - Unusual ports or non-standard protocols.
    - Suspicious, highly entropic, or known-bad DNS queries.
    - Potential reverse shell connections.
    
    Provide your analysis in this structured format:
    1. High-Risk Indicators (if any)
    2. Suspicious Domains/IPs
    3. Recommendations for further investigation
    
    Traffic Summary:
    """

    response = client.models.generate_content(model='gemini-3.5-flash-lite',contents=prompt + extracted_data,config=types.GenerateContentConfig(temperature=0.1 # Keep temperature low to prevent AI hallucinations during security analysis
        )
    )
    
    return response.text


def main():
    file_path = "C:\\Python\\2026-10-01-traffic-analysis-exercise.pcap\\2026-10-01-traffic-analysis-exercise.pcap"
    extracted_data = parse_pcap(file_path)

    #print(extracted_data)
    print("AI enriched analysis")
    print(AI_enrichment(extracted_data))


if __name__ == "__main__":
    main()


AI enriched analysis
### 1. High-Risk Indicators
* **Non-Standard Port Communication (TCP/9000):** The host `10.10.1.128` establishes a TCP connection to `107.175.82.242` on port **9000**. Port 9000 is commonly associated with alternative web services, SonarQube, or more critically, used as a non-standard listener port for command-and-control (C2) frameworks (e.g., Metasploit Meterpreter, custom Python/Go backdoors).
* **Suspicious Domain Resolution:** The presence of `sorrelapse.best`, `holtandberyl.com`, and `www.ars-soar.com` in the DNS query list. These domains do not align with standard corporate infrastructure or common legitimate content delivery networks and exhibit characteristics typical of newly registered domains (NRDs) or domains used in phishing/malware delivery.
* **Internal Active Directory & Kerberos Traffic:** Frequent communication with `10.10.1.10` over ports `389` (LDAP), `88` (Kerberos), and `135`/`445` (RPC/SMB) indicates standard domain-joined workstation activi